# 00 — Setup check

Run this top to bottom. It confirms your environment works and that all 10 ARMD CSVs are reachable — on a **local** machine or on **Colab**. Green `PASS` at the bottom = you're ready.


## 1. Environment bootstrap

On **Colab** this clones the repo, installs deps, mounts Drive, and sets `ARMD_DIR`.
On **local** it does nothing (you already set things up via the README).
Edit the repo URL and the Drive folder name to match yours.

In [ ]:
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("Environment:", "Colab" if IN_COLAB else "Local")

if IN_COLAB:
    # --- clone repo + install (edit the URL) ---
    if not Path("capstone").exists():
        !git clone https://github.com/<your-org>/capstone.git
    %cd capstone
    !pip install -q torch-geometric "flwr[simulation]" pyvis pyarrow
    # --- mount shared Drive + point ARMD_DIR at the shared ARMD folder (edit name) ---
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["ARMD_DIR"] = "/content/drive/MyDrive/ARMD"

# Make src/ importable even without `pip install -e .`
src = Path.cwd() / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

print("ARMD_DIR =", os.environ.get("ARMD_DIR", "(not set — will use data/raw/)"))

## 2. Check the imports

In [ ]:
checks = {}
try:
    import torch; checks["torch"] = torch.__version__
    import torch_geometric; checks["torch_geometric"] = torch_geometric.__version__
    import flwr; checks["flwr"] = flwr.__version__
    import pandas; checks["pandas"] = pandas.__version__
    import sklearn, networkx, matplotlib  # noqa
    checks["others"] = "ok"
    print("Imports OK:")
    for k, v in checks.items():
        print(f"  {k}: {v}")
    print("CUDA available:", torch.cuda.is_available())
    imports_ok = True
except Exception as e:
    print("IMPORT FAILED:", e)
    imports_ok = False

## 3. Check the data path and all 10 CSVs

In [ ]:
from amr_fed import config

print("Looking for ARMD data in:", config.DATA_DIR, "\n")

found, missing = [], []
for key, fname in config.ARMD_TABLES.items():
    p = config.DATA_DIR / fname
    (found if p.exists() else missing).append(fname)

for f in found:   print("  found  ", f)
for f in missing: print("  MISSING", f)

data_ok = len(missing) == 0

## 4. Result

In [ ]:
print("="*44)
if imports_ok and data_ok:
    print("PASS — environment and data are ready.")
elif imports_ok and not data_ok:
    print("PARTIAL — imports OK, but some CSVs are missing.")
    print("Fix ARMD_DIR (cell 1) to point at the shared ARMD folder,")
    print("or wait for Google Drive to finish syncing 'available offline'.")
else:
    print("FAIL — imports broke. Re-check the install steps in the README.")
print("="*44)